In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DATA_DIR = "/content/drive/MyDrive/AI-Study-RAG"
import os
os.listdir(DATA_DIR)

['nlp', 'ml', 'dl', 'cv']

In [6]:
!pip install pypdf pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 6.7 MB/s eta 0:00:00


In [7]:
from pypdf import PdfReader
import pandas as pd

records = []

for domain in os.listdir(DATA_DIR):
    domain_path = os.path.join(DATA_DIR, domain)
    if not os.path.isdir(domain_path):
        continue
    for filename in os.listdir(domain_path):
        if not filename.lower().endswith(".pdf"):
            continue
        filepath = os.path.join(domain_path, filename)
        try:
            reader = PdfReader(filepath)
            num_pages = len(reader.pages)
            sample_text = ""
            for page in reader.pages[:3]:
                sample_text += page.extract_text() or ""
            status = "OK" if len(sample_text.strip()) > 50 else "NEEDS_OCR_OR_EMPTY"
        except Exception as e:
            num_pages = 0
            status = f"FAILED: {e}"

        records.append({
            "domain": domain,
            "filename": filename,
            "num_pages": num_pages,
            "status": status
        })

df_inspect = pd.DataFrame(records)
df_inspect

,domain,filename,num_pages,status
0,nlp,Natural Language Processing.pdf,258,OK
1,ml,Machine_Learning_with_Scikit-Learn.pdf,851,OK
2,dl,Deep learning.pdf,1151,OK
3,cv,Computer Vision.pdf,129,OK


## 2.1 Load & Inspect

Four knowledge domains (ML, DL, NLP, CV) were collected from university textbooks in PDF format:

| Domain | File | Pages | Status |
|--------|------|-------|--------|
| ML | Machine_Learning_with_Scikit-Learn.pdf | 851 | OK |
| DL | Deep learning.pdf | 1151 | OK |
| NLP | Natural Language Processing.pdf | 258 | OK |
| CV | Computer Vision.pdf | 129 | OK |

**Total**: 4 files, 2389 pages.

All files are native text PDFs, fully extractable. No file required OCR or failed during processing.

In [8]:
def split_text(text, chunk_size=1000, chunk_overlap=200):
    """Split text into overlapping chunks of fixed character size."""
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - chunk_overlap  # move forward, keeping overlap

    return chunks

In [10]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

all_chunks = []

for domain in os.listdir(DATA_DIR):
    domain_path = os.path.join(DATA_DIR, domain)
    if not os.path.isdir(domain_path):
        continue
    for filename in os.listdir(domain_path):
        if not filename.lower().endswith(".pdf"):
            continue
        filepath = os.path.join(domain_path, filename)
        reader = PdfReader(filepath)

        full_text = ""
        for page in reader.pages:
            page_text = page.extract_text() or ""
            full_text += page_text + "\n"

        chunks = split_text(full_text, CHUNK_SIZE, CHUNK_OVERLAP)

        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "domain": domain,
                "source_file": filename,
                "chunk_id": f"{domain}_{filename}_{i}",
                "text": chunk
            })

print(f"Total number of chunks: {len(all_chunks)}")

Total number of chunks: 5514


In [11]:
print(all_chunks[100]["source_file"])
print("---")
print(all_chunks[100]["text"])

Natural Language Processing.pdf
---
 real-world problems. 
Modern NLP relies heavily on software libraries, programming languages, and 
development environments that enable efficient implementation of complex algorithms. 
Programming languages such as Python have become the standard in NLP due to their 
simplicity and the availability of powerful libraries. Frameworks and libraries such as 
NLTK, spaCy, Scikit-learn, TensorFlow, and PyTorch provide pre-built components for 
tasks such as tokenization, parsing, machine learning, and deep learning. These tools 
 
 
8 
 
significantly reduce the effort required to build NLP systems from scratch and allow 
researchers and practitioners to focus on higher-level problem solving. 
Practical tools also facilitate experimentation, which is a critical aspect of NLP. 
Language data is inherently variable and often noisy, meaning that models must be 
tested, evaluated, and refined iteratively. Tools for data preprocessing, visualization, and 
eval

In [12]:
df_chunks = pd.DataFrame(all_chunks)
df_chunks.groupby("domain").size()

,0
domain,
cv,119
dl,2847
ml,2132
nlp,416


## 2.2 Chunking Strategy

A fixed-size chunking strategy was used: each document is split into chunks of
**1000 characters** with a **200-character overlap** between consecutive chunks.

**Justification**:
- 1000 characters (~150-200 words) is large enough to preserve a coherent unit of
  meaning without being so large that irrelevant content dilutes retrieval relevance.
- A 200-character overlap ensures information spanning chunk boundaries is not lost.
- This approach is domain-agnostic and works reliably across all four textbooks.

**Result**: 5514 total chunks generated from 2389 pages across 4 domains.

In [13]:
!pip install chromadb sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully


In [15]:
def clean_text(text):
    """Remove surrogate characters and other invalid unicode that break the tokenizer."""
    if not isinstance(text, str):
        text = str(text)
    text = text.encode("utf-8", errors="ignore").decode("utf-8", errors="ignore")
    cleaned = ""
    for ch in text:
        try:
            ch.encode("utf-8")
            cleaned += ch
        except UnicodeEncodeError:
            continue
    return cleaned.strip()

for chunk in all_chunks:
    chunk["text"] = clean_text(chunk["text"])

texts = [chunk["text"] for chunk in all_chunks]
print(f"Total texts: {len(texts)}")
print(f"Empty after cleaning: {sum(1 for t in texts if len(t) == 0)}")

Total texts: 5514
Empty after cleaning: 0


In [16]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

print(f"Generated {len(embeddings)} embeddings, each of dimension {embeddings.shape[1]}")

Batches:   0%|          | 0/173 [00:00<?, ?it/s]

Generated 5514 embeddings, each of dimension 384


In [17]:
import chromadb

chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/AI-Study-RAG/vector_store")
collection = chroma_client.get_or_create_collection(name="ai_study_docs")

print("ChromaDB collection ready")

ChromaDB collection ready


In [18]:
# ChromaDB needs ids, documents (text), embeddings, and metadata as separate lists
ids = [chunk["chunk_id"] for chunk in all_chunks]
documents = [chunk["text"] for chunk in all_chunks]
metadatas = [{"domain": chunk["domain"], "source_file": chunk["source_file"]} for chunk in all_chunks]

# Add in batches to avoid overloading (ChromaDB has a max batch size)
BATCH_SIZE = 500

for i in range(0, len(ids), BATCH_SIZE):
    end = min(i + BATCH_SIZE, len(ids))
    collection.add(
        ids=ids[i:end],
        documents=documents[i:end],
        embeddings=embeddings[i:end].tolist(),
        metadatas=metadatas[i:end]
    )
    print(f"Added {end}/{len(ids)}")

print("\nAll chunks stored in ChromaDB successfully!")
print(f"Total items in collection: {collection.count()}")

Added 500/5514
Added 1000/5514
Added 1500/5514
Added 2000/5514
Added 2500/5514
Added 3000/5514
Added 3500/5514
Added 4000/5514
Added 4500/5514
Added 5000/5514
Added 5500/5514
Added 5514/5514

All chunks stored in ChromaDB successfully!
Total items in collection: 5514


## 2.3 Embeddings & Vector Store

- **Embedding model**: `all-MiniLM-L6-v2` (Sentence-Transformers) — 384-dimensional embeddings.
- **Vector database**: ChromaDB, in persistent mode so the backend loads it directly.
- **Storage location**: `vector_store/`, containing collection `ai_study_docs` with 5514 chunks.

**Note**: Some Unicode surrogate characters from PDF extraction caused encoding errors and were cleaned before generating embeddings.

## 2.4 Retrieval & Prompting

A retrieval function was implemented using cosine similarity search over the ChromaDB
vector store, returning the top-k most relevant chunks for a given query. It was first
tested in isolation (retrieval only), then combined with a prompt template and an
Ollama LLM to produce grounded, cited answers.

In [19]:
def retrieve_chunks(query, n_results=3):
    """Retrieve the most relevant chunks for a given query."""
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "text": results["documents"][0][i],
            "domain": results["metadatas"][0][i]["domain"],
            "source": results["metadatas"][0][i]["source_file"],
            "distance": results["distances"][0][i]
        })
    return retrieved

test_results = retrieve_chunks("What is a convolutional neural network?")
for r in test_results:
    print(f"[{r['domain']} | {r['source']}] (distance: {r['distance']:.4f})")
    print(r['text'][:200])
    print("---")

[ml | Machine_Learning_with_Scikit-Learn.pdf] (distance: 0.7823)
This was made
466 | Chapter 14: Deep Computer Vision Using Convolutional Neural Networks
14 In the 2010 movie Inception, the characters keep going deeper and deeper into multiple layers of dreams;
hen
---
[ml | Machine_Learning_with_Scikit-Learn.pdf] (distance: 0.8017)
ce its cuteness. Nor can you explain how you recog‐
nize a cute puppy; it’s just obvious to you. Thus, we cannot trust our subjective expe‐
rience: perception is not trivial at all, and to understand 
---
[ml | Machine_Learning_with_Scikit-Learn.pdf] (distance: 0.8202)
olutional Networks
The idea of FCNs was first introduced in a 2015 paper25 by Jonathan Long et al., for
semantic segmentation (the task of classifying every pixel in an image according to
the class of
---


In [20]:
retrieval_test_questions = [
    "What is a convolutional neural network?",
    "What is stemming in NLP?",
    "What is linear regression?",
    "What is object detection in computer vision?",
    "What is overfitting in machine learning?",
    "What is tokenization?",
    "What is a recurrent neural network?",
    "What is the difference between supervised and unsupervised learning?",
    "What is image segmentation?",
    "What is a word embedding?"
]

for q in retrieval_test_questions:
    print(f"QUESTION: {q}")
    results = retrieve_chunks(q, n_results=2)
    for r in results:
        print(f"  -> [{r['domain']} | {r['source']}] (distance: {r['distance']:.4f})")
        print(f"     {r['text'][:150]}...")
    print("=" * 80)

QUESTION: What is a convolutional neural network?
  -> [ml | Machine_Learning_with_Scikit-Learn.pdf] (distance: 0.7823)
     This was made
466 | Chapter 14: Deep Computer Vision Using Convolutional Neural Networks
14 In the 2010 movie Inception, the characters keep going dee...
  -> [ml | Machine_Learning_with_Scikit-Learn.pdf] (distance: 0.8017)
     ce its cuteness. Nor can you explain how you recog‐
nize a cute puppy; it’s just obvious to you. Thus, we cannot trust our subjective expe‐
rience: pe...
QUESTION: What is stemming in NLP?
  -> [nlp | Natural Language Processing.pdf] (distance: 0.5204)
     istency in 
representation. 
This makes stemming: 
• Fast and computationally efficient 
• Suitable for large-scale applications 
• Less accurate comp...
  -> [nlp | Natural Language Processing.pdf] (distance: 0.5502)
     word removal should be viewed as a task-dependent preprocessing decision 
rather than a universal rule. 
2.6 Stemming 
2.6.1 Concept and Motivation of...
QUESTION: W

In [21]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (2,481 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video gro

In [22]:
import subprocess
import time

process = subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
print("Ollama server started")

Ollama server started


In [23]:
!ollama pull llama3.2:1b
!pip install ollama -q

In [24]:
import ollama

response = ollama.chat(model='llama3.2:1b', messages=[
    {'role': 'user', 'content': 'Say hello in one sentence.'}
])
print(response['message']['content'])

I'm happy to help you with anything you need.


In [32]:
def build_prompt(question, retrieved_chunks):
    """Build a prompt that grounds the LLM's answer in retrieved context."""
    context = "\n\n".join([
        f"[Source: {c['domain']}/{c['source']}]\n{c['text']}"
        for c in retrieved_chunks
    ])

    prompt = f"""You are a helpful study assistant. Answer the question using ONLY the context below.
If the context does not contain enough information, say so honestly instead of guessing.
Always mention which source(s) you used in your answer.

Context:
{context}

Question: {question}

Answer:"""
    return prompt


def ask_rag(question, n_results=3, distance_threshold=1.0):
    """Full RAG pipeline: retrieve chunks, build prompt, call LLM."""
    retrieved = retrieve_chunks(question, n_results=n_results)

    # Reject if the best match is too far (irrelevant to the knowledge base)
    if not retrieved or retrieved[0]["distance"] > distance_threshold:
        return {
            "question": question,
            "answer": "I don't have enough relevant information in the provided textbooks to answer this question.",
            "sources": []
        }

    prompt = build_prompt(question, retrieved)

    response = ollama.chat(model='llama3.2:1b', messages=[
        {'role': 'user', 'content': prompt}
    ])

    answer = response['message']['content']
    sources = list(set([f"{c['domain']}/{c['source']}" for c in retrieved]))

    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }
result = ask_rag("What is stemming in NLP?")
print("ANSWER:", result["answer"])
print("\nSOURCES:", result["sources"])

ANSWER: According to the provided context, stemming in NLP (Natural Language Processing) refers to the process of reducing words to their root or base form, commonly referred to as the stem. This is a text normalization technique that groups together different inflected or derived forms of a word so that they can be treated as a single unit during analysis.

SOURCES: ['nlp/Natural Language Processing.pdf']


In [34]:
test_questions = [
    "What is a convolutional neural network?",
    "What is stemming in NLP?",
    "What is linear regression?",
    "What is object detection in computer vision?",
    "What is overfitting in machine learning?",
    "What is tokenization?",
    "What is a recurrent neural network?",
    "What is the difference between supervised and unsupervised learning?",
    "What is image segmentation?",
    "What is a word embedding?",
    "What is the boiling point of water?",
    "How do you make a cup of tea?"
]

evaluation_results = []

for idx, q in enumerate(test_questions, start=1):
    result = ask_rag(q)
    evaluation_results.append(result)

    print(f"\n{'-' * 80}")
    print(f"QUESTION {idx}: {result['question']}")
    print(f"{'-' * 80}")
    print(f"\nAnswer:\n{result['answer']}")
    print(f"\nSources: {', '.join(result['sources']) if result['sources'] else 'None (rejected - out of scope)'}")
    print(f"{'-' * 80}\n")


--------------------------------------------------------------------------------
QUESTION 1: What is a convolutional neural network?
--------------------------------------------------------------------------------

Answer:
Based on the provided context, a convolutional neural network (CNN) is a type of neural network that uses convolutional and pooling layers to process images.

I couldn't find any information that suggests the context implies the term "inception modules" or that it has any relation to the 2010 movie Inception. The context primarily discusses the emergence of CNNs from the study of the brain's visual cortex and their success in various visual tasks, such as image recognition and object classification.

Sources: ml/Machine_Learning_with_Scikit-Learn.pdf
--------------------------------------------------------------------------------


--------------------------------------------------------------------------------
QUESTION 2: What is stemming in NLP?
------------------

In [35]:
results_df = pd.DataFrame(evaluation_results)
results_df[["question", "sources", "answer"]]

,question,sources,answer
0,What is a convolutional neural network?,[ml/Machine_Learning_with_Scikit-Learn.pdf],"Based on the provided context, a convolutional..."
1,What is stemming in NLP?,[nlp/Natural Language Processing.pdf],Based on the provided context from [nlp/Natura...
2,What is linear regression?,"[dl/Deep learning.pdf, ml/Machine_Learning_wit...","Based on the provided context, linear regressi..."
3,What is object detection in computer vision?,"[cv/Computer Vision.pdf, dl/Deep learning.pdf]","Based on the provided context, object detectio..."
4,What is overfitting in machine learning?,[ml/Machine_Learning_with_Scikit-Learn.pdf],Based on the provided context from the Machine...
5,What is tokenization?,[nlp/Natural Language Processing.pdf],"According to the provided context, tokenizatio..."
6,What is a recurrent neural network?,"[nlp/Natural Language Processing.pdf, dl/Deep ...",According to [Source: nlp/Natural Language Pro...
7,What is the difference between supervised and ...,[ml/Machine_Learning_with_Scikit-Learn.pdf],"Based on the context provided, the main differ..."
8,What is image segmentation?,[cv/Computer Vision.pdf],"Based on the context provided, image segmentat..."
9,What is a word embedding?,[nlp/Natural Language Processing.pdf],"According to the provided sources, a word embe..."


## 2.5 Vision Component

*Not applicable* — this project follows the **Core Track** (text-only RAG assistant).
The Vision/YOLO component in this section is only required for the Extended Track.

## 2.6 Evaluation

| # | Question | Retrieved Source(s) | Answer Summary | Correct? |
|---|----------|---------------------|-----------------|-----------|
| 1 | What is a CNN? | ml | CNN uses conv & pooling layers for images | ✅ True |
| 2 | What is stemming in NLP? | nlp | Reduces words to root/base form via heuristic rules | ✅ True |
| 3 | What is linear regression? | dl, ml | Single-layer neural network / weighted sum of features | ✅ True |
| 4 | What is object detection? | cv, dl | Identifies & locates objects in images | ✅ True |
| 5 | What is overfitting? | ml | Performs well on train, fails to generalize | ✅ True |
| 6 | What is tokenization? | nlp | Breaking text into words/sentences/subword tokens | ✅ True |
| 7 | What is an RNN? | nlp, dl, ml | Sequence model with recursive hidden state / "memory" | ✅ True |
| 8 | Supervised vs unsupervised? | ml | Labeled vs unlabeled training data | ✅ True |
| 9 | What is image segmentation? | cv | Partitioning an image into constituent parts/objects | ✅ True |
| 10 | What is a word embedding? | nlp | Dense vector representation capturing semantics | ✅ True |
| 11 | What is the boiling point of water? | none (rejected) | Correctly refused — no relevant context found | ✅ True (correct refusal) |
| 12 | How do you make a cup of tea? | none (rejected) | Correctly refused — no relevant context found | ✅ True (correct refusal) |

**Accuracy**: 12/12 correct — 10 fully grounded factual answers from the correct
domains, and 2 out-of-scope questions correctly rejected using a retrieval
distance threshold, with no hallucination.

### Main Failure Cases (observed during development)
- **Semantic-similarity false positive**: In earlier testing, a question about the
  "capital of France" retrieved NLP chunks discussing word-embedding arithmetic
  examples (which coincidentally mention "France" and "Paris" as illustrative
  tokens), causing the LLM to hallucinate an incorrect answer.
- **Unrestricted general-knowledge leakage**: Before adding a distance threshold,
  out-of-scope factual questions (e.g. "boiling point of water") were sometimes
  answered from the LLM's own general knowledge rather than being rejected — a
  direct violation of proper RAG grounding.
- **Hallucinated attribution**: In some runs, the model invented details (author
  names, citation URLs) not present in the retrieved context, even when the core
  answer was otherwise correct.

### Mitigation
- Strengthened the prompt to explicitly instruct the model to rely only on the
  provided context.
- **Added a retrieval distance threshold** (`distance_threshold = 1.0`) to
  `ask_rag()`: if the closest retrieved chunk exceeds this distance, the system
  returns a fixed "not enough information" response instead of calling the LLM at
  all. This eliminated hallucination on both out-of-scope test questions
  (questions 11–12).

In [30]:
import json

config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_model": "all-MiniLM-L6-v2",
    "embedding_dimension": 384,
    "vector_store_path": "vector_store",
    "collection_name": "ai_study_docs",
    "total_chunks": len(all_chunks),
    "domains": list(df_chunks["domain"].unique().tolist()),
    "llm_model": "llama3.2:1b"
}

config_path = "/content/drive/MyDrive/AI-Study-RAG/vector_store/config.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Config saved successfully:")
print(json.dumps(config, indent=2))

Config saved successfully:
{
  "chunk_size": 1000,
  "chunk_overlap": 200,
  "embedding_model": "all-MiniLM-L6-v2",
  "embedding_dimension": 384,
  "vector_store_path": "vector_store",
  "collection_name": "ai_study_docs",
  "total_chunks": 5514,
  "domains": [
    "nlp",
    "ml",
    "dl",
    "cv"
  ],
  "llm_model": "llama3.2:1b"
}


In [31]:
vector_store_dir = "/content/drive/MyDrive/AI-Study-RAG/vector_store"
print("Contents of vector_store directory:")
for item in os.listdir(vector_store_dir):
    print(" -", item)

Contents of vector_store directory:
 - chroma.sqlite3
 - 9806072d-da89-4ea7-bfbd-f94e143acdc8
 - config.json


## 2.7 Export

The persisted vector store (ChromaDB) and a `config.json` file were saved to
`vector_store/`. This allows the backend to load the vector store directly at
startup without recomputing embeddings or rebuilding the index.